# AegisLLM — laboratório executável de AI Security

**Público:** pessoas estudando segurança de LLMs, AppSec e LLMOps.  
**Pré-requisitos:** Python 3.10+ e o repositório local. Nenhuma chave de API é necessária.  
**Objetivos:** executar o gateway, observar decisões de segurança, comparar o roteamento, rodar red team, validar gates e persistir auditoria sem prompts brutos.

Roteiro: pipeline; fluxo permitido; ataques; roteamento; red team; benchmark/gates; Security Agents; auditoria.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "aegis").exists():
    ROOT = ROOT.parent
if not (ROOT / "aegis").exists():
    raise RuntimeError("Execute dentro do repositório AegisLLM")
sys.path.insert(0, str(ROOT))

from aegis import AegisGateway, Request
from aegis.classification import classify
from aegis.storage import LocalRepository
from evaluation import run_benchmark
from evaluation.gates import evaluate_gates
from redteam.runner import run_red_team, summarize
from security_agents import SecurityOrchestrator


def show_table(rows):
    if not rows:
        print("(sem dados)")
        return
    headers = list(rows[0])
    try:
        from IPython.display import HTML, display

        head = "".join(f"<th>{header}</th>" for header in headers)
        body = "".join(
            "<tr>" + "".join(f"<td>{row.get(header, '')}</td>" for header in headers) + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(headers)
        for row in rows:
            print([row.get(header, "") for header in headers])


gateway = AegisGateway(rate_limit=1000)
print(f"Projeto carregado: {ROOT}")

## 1. Pipeline integrado

Cada etapa pode bloquear a requisição e registra somente metadados minimizados.

In [ ]:
stages = ["API/Auth", "Classificação + DLP", "Guardrails/RAG", "Policy", "Model Router", "Secure Tools", "Output Validation", "Audit"]
print("  →  ".join(stages))

## 2. Fluxo permitido e tenant isolation

In [ ]:
request = Request("tenant-a", "ana", "support", "Verifique minha última compra", requested_tool="order_read")
response = gateway.handle(request)
show_table([{"status": response.status, "modelo": response.model, "risco": response.risk_level, "tool_result": response.tool_result, "classification": response.metadata["classification"]}])
assert response.status == "allowed" and response.tool_result == {"last_order": "Pedido A-100"}

## 3. Ataques demonstráveis

Injection direta/indireta, exfiltração, abuso de ferramenta, cross-tenant, saída ativa, jailbreak e excessive agency.

In [ ]:
findings = run_red_team(AegisGateway(rate_limit=1000))
show_table([{k: item[k] for k in ("name", "category", "owasp_id", "severity", "attack_success")} for item in findings])
summary = summarize(findings)
print("Resumo:", summary)
assert summary["attack_success_rate"] == 0.0

## 4. Roteamento multiobjetivo e restrições obrigatórias

In [ ]:
router = gateway.router
rows = []
for prompt, task in [("Resuma o pedido", "customer_support"), ("Analise risco crítico", "critical"), ("Minha senha é secret-123", "customer_support")]:
    classification, _ = classify(prompt)
    req = Request("tenant-a", "router-demo", "analyst", prompt, task=task)
    chosen = router.choose(req, classification)
    rows.append({"tarefa": task, "classificação": classification, "modelo": chosen.name, "provider": chosen.provider, "utility": round(router.utility(chosen, req), 3)})
show_table(rows)
print("Fronteira de Pareto:", [model.name for model in router.pareto_frontier()])
assert rows[-1]["provider"] == "local"

## 5. Benchmark reproduzível e security gates

In [ ]:
benchmark = run_benchmark()
gates = evaluate_gates(benchmark)
show_table(benchmark["results"])
print({k: v for k, v in benchmark.items() if k != "results"})
print("Gates:", gates)
assert gates["passed"]

## 6. Security Agents: Attack → Judge → Defense → Triage

In [ ]:
agent_report = SecurityOrchestrator().run()
show_table(agent_report["triage"])
show_table(agent_report["defense"][:4])
assert all(judgment["passed"] for judgment in agent_report["judgments"])

## 7. Auditoria com minimização de dados

In [ ]:
audit_gateway = AegisGateway()
audit_response = audit_gateway.handle(Request("tenant-a", "auditor", "support", "Meu email é pessoa@example.com"))
repository = LocalRepository()
repository.record_audit(audit_response.metadata)
print("Campos auditados:", sorted(audit_response.metadata))
print("Eventos persistidos:", repository.count("audit_events"))
assert "prompt" not in audit_response.metadata and "pessoa@example.com" not in str(audit_response.metadata)

## Exercício

Adicione um novo `AttackCase` e confirme que o gate continua passando. Use apenas o laboratório local e dados fictícios.

**Armadilha comum:** comparar Attack Success Rate com corpus, orçamento ou juiz diferentes. Preserve essas condições antes de comparar baselines.

**Extensão:** habilite Promptfoo/PyRIT/Garak somente contra o container local e normalize a saída no formato `Finding`.

In [ ]:
# Scaffold: execute run_red_team(cases=[seu_caso]) e avalie o resultado.
print("Notebook concluído com sucesso.")